# Part 3 – YOLO Training & Evaluation

**Branch:** `feature/yolo-training-evaluation`


## 7. Train YOLO11s

In [ ]:
EPOCHS = 60
IMGSZ = 640
shutil.rmtree('/content/runs/yolo11s', ignore_errors=True)   # re-runs overwrite instead of creating yolo11s2
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs={EPOCHS} imgsz={IMGSZ} \
    project=/content/runs name=yolo11s

In [ ]:
YOLO_WEIGHTS = '/content/runs/yolo11s/weights/best.pt'
assert os.path.exists(YOLO_WEIGHTS), "YOLO11s training did not produce best.pt — check the training log above."
print("YOLO11s weights:", YOLO_WEIGHTS)

### 7b. Training curves

In [ ]:
yolo_results_csv = '/content/runs/yolo11s/results.csv'
if os.path.exists(yolo_results_csv):
    yolo_hist = pd.read_csv(yolo_results_csv)
    yolo_hist.columns = [c.strip() for c in yolo_hist.columns]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(yolo_hist['epoch'], yolo_hist['train/box_loss'], label='train/box_loss')
    axes[0].plot(yolo_hist['epoch'], yolo_hist['val/box_loss'], label='val/box_loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('YOLO11s -- Box Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(yolo_hist['epoch'], yolo_hist['metrics/mAP50(B)'], label='mAP50')
    axes[1].plot(yolo_hist['epoch'], yolo_hist['metrics/mAP50-95(B)'], label='mAP50-95')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP')
    axes[1].set_title('YOLO11s -- Validation mAP'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/yolo11s_training_curves.png', dpi=150)
    plt.show()
else:
    print(f"{yolo_results_csv} not found -- run the training cell first.")

## 8. Evaluate YOLO11s

### 8a. Validation split

In [ ]:
METRIC_COLS = ['model', 'mAP50', 'mAP50-95', 'precision', 'recall', 'f1']

def evaluate_yolo(model_path, model_name, data_yaml, split):
    """Run Ultralytics validation on one split ('val' or 'test'). Returns a dict of metrics and the output dir."""
    model = YOLO(model_path)
    results = model.val(data=data_yaml, split=split)
    p, r = results.box.mp, results.box.mr
    return {
        'model': model_name, 'split': split,
        'mAP50': results.box.map50, 'mAP50-95': results.box.map,
        'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r + 1e-9),
        'save_dir': str(results.save_dir),
    }

yolo_val_metrics = evaluate_yolo(YOLO_WEIGHTS, 'YOLO11s', '/content/data.yaml', 'val')
comparison_df = pd.DataFrame([yolo_val_metrics])[METRIC_COLS]
comparison_df

### 8b. Test split (final, evaluated once)

In [ ]:
yolo_test_metrics = evaluate_yolo(YOLO_WEIGHTS, 'YOLO11s', '/content/data.yaml', 'test')
comparison_test_df = pd.DataFrame([yolo_test_metrics])
comparison_test_df

### 8c. Test metrics chart

In [ ]:
metric_cols = ['mAP50', 'mAP50-95', 'precision', 'recall', 'f1']
metrics_series = comparison_test_df.set_index('model')[metric_cols].iloc[0]

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#4C72B0', '#8CA9D6', '#DD8452', '#55A868', '#C44E52']
bars = ax.bar(metrics_series.index, metrics_series.values, color=bar_colors)
plt.title('YOLO11s -- Detection Performance (TEST split, touched once)')
plt.ylabel('Score')
plt.xticks(rotation=0, fontsize=11)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3, axis='y')
for b, v in zip(bars, metrics_series.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.2f}', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('/content/model_comparison_test.png', dpi=150)
plt.show()

### 8d. Test confusion matrix and curves

In [ ]:
DET_PLOTS = ['confusion_matrix.png', 'confusion_matrix_normalized.png',
             'BoxPR_curve.png', 'PR_curve.png', 'BoxF1_curve.png', 'F1_curve.png']   # file names differ across Ultralytics versions

print(f"YOLO11s -- TEST split (from {yolo_test_metrics['save_dir']}):")
for fname in DET_PLOTS:
    fpath = os.path.join(yolo_test_metrics['save_dir'], fname)
    if os.path.exists(fpath):
        display(IPyImage(filename=fpath))

## 9. Detector Input / Output Gallery

Your single uploaded demo pair is reused in every gallery below: detector input → YOLO output → matching input/output → per-object change events → end-to-end pipeline trace.


### 9a. Input: raw reference / query pairs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].imshow(demo_reference_raw); axes[0].axis('off')
axes[0].set_title('Reference -- demo pair', fontsize=12)
axes[1].imshow(demo_query_raw); axes[1].axis('off')
axes[1].set_title('Query -- demo pair', fontsize=12)

plt.suptitle('Gallery: Raw Input Pair (Reference / Query)', fontsize=15, y=1.03)
plt.tight_layout()
plt.savefig('/content/gallery_1_raw_input.png', dpi=150, bbox_inches='tight')
plt.show()

### 9b. Output: YOLO detections (run on the CLAHE-preprocessed version of the same pairs)

In [ ]:
def draw_labeled_box(img, x1, y1, x2, y2, text, color, box_thickness=None):
    """Draw a bounding box plus a filled label chip behind the text, so the class
    name and confidence score stay clearly readable over any background. Box
    thickness, font size and padding all scale with the image resolution, so
    labels stay legible on both small crops and full-size phone-camera photos."""
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    h, w = img.shape[:2]
    scale = max(h, w) / 1000.0

    if box_thickness is None:
        box_thickness = max(4, round(scale * 4))
    cv2.rectangle(img, (x1, y1), (x2, y2), color, box_thickness)

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = max(0.9, scale * 1.1)
    font_thickness = max(2, round(scale * 2.4))
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, font_thickness)
    pad = max(7, round(scale * 7))
    label_y1 = y1 - th - 2 * pad if y1 - th - 2 * pad > 0 else y1  # flip below box if no room above
    label_y2 = label_y1 + th + 2 * pad
    label_x2 = min(x1 + tw + 2 * pad, w - 1)
    cv2.rectangle(img, (x1, label_y1), (label_x2, label_y2), color, -1)
    text_color = (255, 255, 255) if sum(color) < 400 else (0, 0, 0)
    cv2.putText(img, text, (x1 + pad, label_y2 - pad), font, font_scale, text_color,
                font_thickness, cv2.LINE_AA)

yolo_model = YOLO(YOLO_WEIGHTS)

def _yolo_annotate_img(img_bgr, model, conf=0.4):
    """Run YOLO on an in-memory BGR image (the CLAHE-preprocessed demo pair) and return an annotated RGB copy."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    result = model(img_bgr, conf=conf, verbose=False)[0]
    annotated = img_rgb.copy()
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls_name = model.names[int(box.cls[0])]
        conf_score = float(box.conf[0])
        draw_labeled_box(annotated, x1, y1, x2, y2, f'{cls_name} {conf_score:.2f}', (255, 0, 0))
    return annotated, len(result.boxes)

demo_reference_annot, demo_n_reference = _yolo_annotate_img(demo_reference_pre, yolo_model)
demo_query_annot,  demo_n_query  = _yolo_annotate_img(demo_query_pre,  yolo_model)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].imshow(demo_reference_annot); axes[0].axis('off')
axes[0].set_title(f"YOLO Output -- Reference  ({demo_n_reference} box(es), demo pair)", fontsize=12)
axes[1].imshow(demo_query_annot); axes[1].axis('off')
axes[1].set_title(f"YOLO Output -- Query  ({demo_n_query} box(es), demo pair)", fontsize=12)

plt.suptitle('YOLO Output: Detected Boxes on the Demo Pair (class + confidence labeled)',
             fontsize=15, y=1.03)
plt.tight_layout()
plt.savefig('/content/gallery_2_yolo_output.png', dpi=150, bbox_inches='tight')
plt.show()